# 라이브러리 호출

In [ ]:
import pandas as pd 
import os
import datetime
import holidays

# 지하철 데이터 처리

## 지하철 데이터 출처

지하철 역 위치 정보: https://t-data.seoul.go.kr/dataprovide/trafficdataviewfile.do?data_id=36  
지하철 승하차 정보: https://data.seoul.go.kr/dataList/OA-12914/S/1/datasetView.do#

## CSV 디코딩 오류 해결 및 데이터 분리

CSV 파일을 바로 이용하려고 했을 때 읽어오지 못하는 오류가 생겨 이를 해결 및 원본 데이터와 실제로 가공할 데이터를 분리하기 위한 작업을 실행

In [ ]:
# 원래 데이터에서 실제 사용 데이터 쪽으로 다시 저장(디코딩 오류 해결)

# 디렉터리 경로 설정
input_dir = '데이터(보존)/지하철/'
output_dir = '데이터/지하철/'

# 파일명 조건에 맞는 리스트 생성
file_list = [
    f for f in os.listdir(input_dir)
]

for file in file_list:
    input_path = input_dir + file
    output_path = output_dir + file

    try:
        df = pd.read_csv(input_path)
    except UnicodeDecodeError:
        df = pd.read_csv(input_path, encoding='cp949')

    df.to_csv(output_path, index=False, encoding='utf-8')

## 지하철 승하차 데이터 분석 및 처리 후 통합

승하차 데이터 분석을 위해 승하차 데이터를 먼저 확인

In [ ]:
#지하철 데이터 분석
input_dir = '데이터/지하철/'

# 파일명 조건에 맞는 리스트 생성
file_list = [
    f for f in os.listdir(input_dir)
    if f.startswith('CARD_SUBWAY_MONTH') and f.endswith('.csv')
]

for file in file_list:
    input_path = input_dir + file

    df = pd.read_csv(input_path)
    
    print(file + '\n')
    print(df)
    print(df.dtypes)
    print()

그리고 승하차 데이터를 확인했을 때 다음과 같은 문제 발견

문제 1: 데이터가 밀려있는 경우  
이때 사용일자 즉 해당일자는 등록일자 -3의 값으로 계산되어야 함  
그리고 이를 처리하고 등록일자를 제거하는 과정까지 전부 진행
그리고 이를 제거한 csv파일을 모두 통합

문제 2: 데이터의 row 개수가 다름  
이는 실제로 달 별로 역의 개수가 다른 문제가 있음  
먼저 모든 기간에 대하여 지하철 승하차 데이터를 통합한 뒤   
전체 누적 역의 개수를 구하고 이를 통해서 생략된 역의 개수를 달 별로 구한 뒤 추가 처리해야함  

이때 문제 1을 해결을 해야 모든 기간에 대한 지하철 승하차 데이터의 통합이 가능하고 이를 통해 문제 2를 해결 가능하므로 문제 1을 먼저 해결해야 함

문제 1 해결을 위해 일단 승하차 데이터들의 값을 먼저 처리

In [ ]:
# 지하철 승하차 데이터 처리 함수(사용일자, 등록일자 처리 및 실제 사용일자만 남기도록)
def shift_row_right(df):
    # '사용일자'가 8자리 숫자가 아닌 경우 → 데이터가 밀렸다고 판단
    mask = ~df['사용일자'].astype(str).apply(lambda x: x.isdigit() and len(x) == 8)

    if mask.any(): # 데이터가 밀린 CSV 파일
        df_shifted = df.copy()

        # 한 칸씩 오른쪽으로 shift
        df_shifted.loc[mask, '등록일자'] = df.loc[mask, '하차총승객수'].astype(str)
        df_shifted.loc[mask, '하차총승객수'] = df.loc[mask, '승차총승객수'].astype(str)
        df_shifted.loc[mask, '승차총승객수'] = df.loc[mask, '역명']
        df_shifted.loc[mask, '역명'] = df.loc[mask, '노선명']
        df_shifted.loc[mask, '노선명'] = df.loc[mask, '사용일자']

        # 등록일자 datetime 변환
        df_shifted['등록일자'] = pd.to_datetime(df_shifted['등록일자'], format='%Y%m%d', errors='coerce')

        # 사용일자 복원: 등록일자 - 3일
        df_shifted.loc[mask, '사용일자'] = df_shifted.loc[mask, '등록일자'] - pd.Timedelta(days=3)

        # 사용일자, 등록일자 → yyyymmdd 정수형으로 변환
        df_shifted['사용일자'] = pd.to_datetime(df_shifted['사용일자'], errors='coerce').dt.strftime('%Y%m%d').astype(int)
        df_shifted['등록일자'] = df_shifted['등록일자'].dt.strftime('%Y%m%d').astype(int)

        # 승하차총승객수 → int 변환
        df_shifted['승차총승객수'] = pd.to_numeric(df_shifted['승차총승객수'], errors='coerce').astype(int)
        df_shifted['하차총승객수'] = pd.to_numeric(df_shifted['하차총승객수'], errors='coerce').astype(int)

        df_shifted = df_shifted.drop(columns=['등록일자']) # 실제 날짜인 사용일자만 남기고 등록일자 삭제

        return df_shifted
    
    else:
        df = df.drop(columns=['등록일자']) # 실제 날짜인 사용일자만 남기고 등록일자 삭제
        return df

In [ ]:
# 실제로 지하철 승하차 데이터 처리 및 통합(사용일자, 등록일자 처리 및 실제 사용일자만 남기도록 하고)
input_dir = '데이터/지하철/'
all_processed = [] 

# 파일명 조건에 맞는 리스트 생성
file_list = [
    f for f in os.listdir(input_dir)
    if f.startswith('CARD_SUBWAY_MONTH') and f.endswith('.csv')
]

for file in file_list:
    input_path = input_dir + file

    df = pd.read_csv(input_path)

    df_processed = shift_row_right(df)

    # 누적 리스트에 추가
    all_processed.append(df_processed)

# 모든 처리된 DataFrame을 하나로 병합
df_total = pd.concat(all_processed, ignore_index=True)

# 최종 누적 CSV 저장
df_total.to_csv('데이터/지하철/통합_지하철_승하차_데이터.csv', index=False, encoding='utf-8')

In [ ]:
# 통합 지하철 승하차 데이터 로드
df = pd.read_csv('데이터/지하철/통합_지하철_승하차_데이터.csv')

# 데이터 타입 및 결측치 확인
print(df.dtypes)
print()
print(df.isnull().sum())
print()
print(df)

결측치 및 데이터 확인 이후 통합 승하차 데이터가 정상적으로 병합이 되었음을 확인하였음  
추가로 문제 2를 해결해야 함

In [ ]:
# 통합 데이터 로드
df = pd.read_csv('데이터/지하철/통합_지하철_승하차_데이터.csv')

# 사용일자 datetime 변환 및 일자 컬럼 생성
df['사용일자'] = pd.to_datetime(df['사용일자'], format='%Y%m%d', errors='coerce')
df['일자'] = df['사용일자'].dt.date  # datetime.date 형식으로 일자 추출

# 전체 역명 집합
all_stations = set(df['역명'].dropna().unique())

# 누락된 역명을 누적 저장할 집합
cumulative_missing_stations = set()

# 일자별 누락 역명 확인
for day, group in df.groupby('일자'):
    current_stations = set(group['역명'].dropna().unique())
    missing_stations = all_stations - current_stations
    cumulative_missing_stations.update(missing_stations)

# 결과 출력
print("누적 누락된 역 수:", len(cumulative_missing_stations))
print("누적 누락된 역명 목록:")
print(sorted(cumulative_missing_stations))

누적된 누락 역명에 대해서 추가로 구해서 조사를 실시

조사 결과 다음과 같음

관악산(서울대) -> 2022년 5월 28일 개업  
굴포천 -> 조사가 누락되는 경우 O  
까치울 -> 조사가 누락되는 경우 O  
당곡 -> 2022년 5월 28일 개업  
뚝섬유원지, 자양(뚝섬한강공원)  -> 자양(뚝섬한강공원)으로 역명 변경  
보라매공원 -> 2022년 5월 28일 개업  
보라매병원 -> 2022년 5월 28일 개업  
부천시청 -> 조사가 누락되는 경우 O  
부천종합운동장 -> 조사가 누락되는 경우 O  
부평구청  -> 조사가 누락되는 경우 O  
삼산체육관  -> 조사가 누락되는 경우 O  
상동 -> 조사가 누락되는 경우 O  
상봉, 상봉(시외버스터미널) -> 상봉으로 역명 변경  
서울대벤처타운 -> 2022년 5월 28일 개업  
서울지방병무청 -> 2022년 5월 28일 개업  
서원 -> 2022년 5월 28일 개업  
성남 -> 2024년 3월 30일 개업  
소요산 -> 조사가 누락되는 경우 O
신중동 -> 조사가 누락되는 경우 O  
암사역사공원 -> 2024년 8월 10일 개업  
연천 -> 2023년 12월 16일 개업  
운천 -> 2022년 12월 17일 개업  
원종 -> 2023년 7월 1일 개업  
임진강 -> 조사가 누락되는 경우 O
전곡 -> 2023년 12월 16일 개업  
청산, 초성리 -> 2023년 12월 16일 개업, 1월에 초성리에서 청산으로 역명 변경  
춘의 -> 조사가 누락되는 경우 O  
한국항공대, 화전 -> 한국항공대로 역명 변경  

세가지 카테고리  
1. 새로 역이 생긴 경우  
2. 조사가 일별로 누락된 경우  
3. 역명이 변경된 경우  

카테고리 별 대응 방식  
1. 새로 역이 생긴 경우: 모든 날짜에 대하여 역에 대한 행을 만들고 그에 대한 승객수 데이터를 0으로 채워 넣음  
2. 조사가 일별로 누락된 경우: 그 역에 대한 데이터를 전체적으로 삭제  
3. 역명이 변경된 경우: 최신 이름으로 다 변경 및 통합  

그리고 추가로 동일 역명에 대하여 다른 노선명을 가진 경우가 있어 이에 대해서 동일 역명에 대한 모든 노선의 승객수는 합으로 계산하여 통합

In [ ]:
# 데이터 불러오기
df = pd.read_csv("데이터/지하철/통합_지하철_승하차_데이터.csv")

# 신규 개업 역 (역명: 개업일)
new_stations = {
    "관악산(서울대)": "20220528",
    "당곡": "20220528",
    "보라매공원": "20220528",
    "보라매병원": "20220528",
    "서울대벤처타운": "20220528",
    "서울지방병무청": "20220528",
    "서원": "20220528",
    "성남": "20240330",
    "암사역사공원": "20240810",
    "연천": "20231216",
    "운천": "20221217",
    "원종": "20230701",
    "전곡": "20231216",
    "청산": "20231216",
    "초성리": "20231216"
}

# 조사 누락된 역 제거
stations_to_delete = [
    "굴포천", "까치울", "부천시청", "부천종합운동장",
    "부평구청", "삼산체육관", "상동", "소요산", "신중동", "임진강", "춘의"
]

df = df[~df["역명"].isin(stations_to_delete)]

# 역명 통합
renamed_stations = {
    "뚝섬유원지": "자양(뚝섬한강공원)",
    "상봉(시외버스터미널)": "상봉",
    "초성리": "청산",
    "화전": "한국항공대"
}
df["역명"] = df["역명"].replace(renamed_stations)

# 사용일자 목록 확보 (중복 없이 전체 날짜 확보)
date_list = sorted(df["사용일자"].unique())

# 개업 전 일자에 대해 해당 역 추가 (0으로)
rows_to_add = []
for station, open_date in new_stations.items():
    open_date = int(open_date)
    for date in date_list:
        if date < open_date:
            rows_to_add.append({
                "사용일자": date,
                "노선명": "미상",  # 필요 시 정확한 노선명 지정 가능
                "역명": renamed_stations.get(station, station),
                "승차총승객수": 0,
                "하차총승객수": 0
            })

# 원본 + 생성 데이터 결합
df = pd.concat([df, pd.DataFrame(rows_to_add)], ignore_index=True)

# 동일 날짜-노선-역명 조합 통합
df = df.groupby(["사용일자", "역명"], as_index=False).agg({
    "승차총승객수": "sum",
    "하차총승객수": "sum"
})

# 정제 완료된 데이터 저장
df.to_csv("데이터/지하철/통합_지하철_승하차_데이터.csv", index=False)

In [ ]:
# 통합 승하차 데이터 로드
df = pd.read_csv('데이터/지하철/통합_지하철_승하차_데이터.csv')

# 사용일자별로 역명 개수 세기
station_count_per_date = df.groupby('사용일자')['역명'].nunique()

# 고유한 역명 개수 리스트 추출
unique_counts = station_count_per_date.unique()

# 결과 출력
if len(unique_counts) == 1:
    print(f"모든 사용일자에 대해 역명 개수가 동일합니다: {unique_counts[0]}개")
else:
    print("사용일자별로 역명 개수가 서로 다릅니다.")

먼저 확인한 역명들에 대해서 이미 처리 했음에도 추가로 결함 발생을 확인  
이에 따라 추가 분석 실시

In [ ]:
# 기준이 되는 정상적인 역명 개수
expected_count = station_count_per_date.mode()[0]  # 가장 흔한 값으로 자동 설정

# 역명 개수가 기준과 다른 날짜 추출
anomalous_dates = station_count_per_date[station_count_per_date != expected_count]

# 결과 출력
print(f"\n역명 개수가 {expected_count}이 아닌 사용일자 목록:")
print(anomalous_dates.sort_index())

# 전체 기간 동안 존재한 역명 목록 (기준이 될 전체 집합)
all_station_names = set(df['역명'].unique())

# 사용일자별 역명 목록 생성
date_station_map = df.groupby('사용일자')['역명'].unique()

# 누락된 역명 추출
missing_stations_per_date = {}

for date, stations in date_station_map.items():
    missing = all_station_names - set(stations)
    if missing:
        missing_stations_per_date[date] = sorted(missing)

# 결과 출력
print(f"\n누락된 역명이 존재하는 날짜 수: {len(missing_stations_per_date)}\n")

for date in sorted(missing_stations_per_date.keys()):
    print(f"{date} - 누락된 역 {len(missing_stations_per_date[date])}개:")
    print(", ".join(missing_stations_per_date[date]))
    print()

이를 통해서 연천, 운천, 전곡, 청산의 경우에도 조사가 누락된 부분이 있었는지를 추가 확인  
이들 역에 데이터에 대해서도 전체적으로 삭제

In [ ]:
# 통합 승하차 데이터 로드
df = pd.read_csv('데이터/지하철/통합_지하철_승하차_데이터.csv')

# 조사 누락된 역 제거
stations_to_delete = [
    "연천", "운천", "전곡", "청산"
]

df = df[~df["역명"].isin(stations_to_delete)].reset_index(drop=True)

df.to_csv("데이터/지하철/통합_지하철_승하차_데이터.csv", index=False)

print(df.dtypes)
print("\n결측치 확인\n", df.isnull().sum())
print()
print(df)

In [ ]:
# 통합 승하차 데이터 로드
df = pd.read_csv('데이터/지하철/통합_지하철_승하차_데이터.csv')

# 사용일자별로 고유 역명 개수 계산
station_count_per_date = df.groupby('사용일자')['역명'].nunique()

# 고유한 역명 개수 리스트 추출
unique_counts = station_count_per_date.unique()

# 결과 출력
if len(unique_counts) == 1:
    print(f"모든 사용일자에 대해 역명 개수가 동일합니다: {unique_counts[0]}개")
else:
    print("사용일자별로 역명 개수가 서로 다릅니다.")
    print("일자별 고유 역명 수:")
    print(station_count_per_date.sort_index())

# 사용일자-역명 기준으로 중복되는 경우 확인
duplicate_check = df.groupby(['사용일자', '역명']).size().reset_index(name='count')
duplicates = duplicate_check[duplicate_check['count'] > 1]

# 중복 여부 출력
if not duplicates.empty:
    print(f"\n중복된 (사용일자, 역명) 조합이 {len(duplicates)}건 존재합니다.")
    print("중복 목록:")
    print(duplicates.sort_values(['사용일자', '역명']))
else:
    print("\n모든 (사용일자, 역명) 조합이 고유합니다. 중복 없음.")

print(df.dtypes)
print("\n결측치 확인\n", df.isnull().sum())
print()
print(df)

역명 개수 및 결측치 확인을 통해 지하철 승하차 데이터의 통합 및 처리가 정상적으로 이루어졌음을 확인  
그러나 569974/521 = 1094 인데 2022-01-01 ~ 2024-12-31 까지는 1096일임  
이에 따라서 빠진 일자를 찾아야함

In [ ]:
# CSV 파일 불러오기
df = pd.read_csv('데이터/지하철/통합_지하철_승하차_데이터.csv')

# '사용일자' 컬럼에서 고유 날짜 추출
existing_dates = set(df['사용일자'].unique())

# 전체 날짜 생성: 2022-01-01 ~ 2024-12-31, 정수형(YYYYMMDD)으로 변환
full_dates = pd.date_range(start='2022-01-01', end='2024-12-31', freq='D')
full_dates_int = set(int(d.strftime('%Y%m%d')) for d in full_dates)

# 빠진 날짜 식별
missing_dates = sorted(full_dates_int - existing_dates)

print(missing_dates)

빠진 일자는 2023-06-26과 2024-07-05임을 파악 완료  
이때 빠진 일자에 대해서 처리하기 위해 다양한 방법을 생각함  
1. 0으로 채우는 방법  
2. 그 역에 대한 전체 평균으로 일자를 채우는 방법  
3. 가까운 동일 요일로 일자를 채우는 방법  
4. 아무것도 채우지 않는 방법  

그 데이터와 최대한 유사하면서 나중에 데이터 분석에 용이하기 위해서는 3안이 최적이라고 판단  
빈값을 채우기 위해서 그 전주의 데이터를 임의로 삽입 

In [ ]:
# 데이터 불러오기
df = pd.read_csv('데이터/지하철/통합_지하철_승하차_데이터.csv')

# 대상 누락 날짜
missing_dates = [20230626, 20240705]

# 각 누락 날짜에 대해 일주일 전 데이터 복제하여 삽입
for missing_date in missing_dates:
    prev_week_date = pd.to_datetime(str(missing_date), format='%Y%m%d') - pd.Timedelta(days=7)
    prev_week_int = int(prev_week_date.strftime('%Y%m%d'))
    
    # 일주일 전 데이터 추출
    ref_data = df[df['사용일자'] == prev_week_int].copy()
    
    if not ref_data.empty:
        # 날짜만 누락된 날짜로 변경
        ref_data['사용일자'] = missing_date
        df = pd.concat([df, ref_data], ignore_index=True)
    else:
        print(f"{prev_week_int}에 해당하는 데이터가 없습니다. 삽입 불가.")

# 날짜 순으로 정렬 (선택 사항)
df = df.sort_values(by='사용일자').reset_index(drop=True)

In [ ]:
# 사용일자별로 고유 역명 개수 계산
station_count_per_date = df.groupby('사용일자')['역명'].nunique()

# 고유한 역명 개수 리스트 추출
unique_counts = station_count_per_date.unique()

# 결과 출력
if len(unique_counts) == 1:
    print(f"모든 사용일자에 대해 역명 개수가 동일합니다: {unique_counts[0]}개")
else:
    print("사용일자별로 역명 개수가 서로 다릅니다.")
    print("일자별 고유 역명 수:")
    print(station_count_per_date.sort_index())

# ✅ 사용일자-역명 기준으로 중복되는 경우 확인
duplicate_check = df.groupby(['사용일자', '역명']).size().reset_index(name='count')
duplicates = duplicate_check[duplicate_check['count'] > 1]

# 중복 여부 출력
if not duplicates.empty:
    print(f"\n중복된 (사용일자, 역명) 조합이 {len(duplicates)}건 존재합니다.")
    print("중복 목록:")
    print(duplicates.sort_values(['사용일자', '역명']))
else:
    print("\n모든 (사용일자, 역명) 조합이 고유합니다. 중복 없음.")

print(df.dtypes)
print("\n결측치 확인\n", df.isnull().sum())
print()
print(df)

df.to_csv("데이터/지하철/통합_지하철_승하차_데이터.csv", index=False)

결측치 및 역명 개수 그리고 571016/521 = 1096 이므로 데이터 정상적으로 가공을 확인

## 통합 지하철 승하차 데이터 & 지하철 위치 데이터 통합 및 처리

In [ ]:
# 지하철 위치 데이터 로드
df = pd.read_csv('데이터/지하철/지하철역_GEOM (역사마스터).csv')

# 데이터 확인
print(df.dtypes)
print("\n결측치 확인\n", df.isnull().sum())
print(df)

지하철 위치 데이터를 확인  

지하철 위치 데이터와 지하철 승하차 데이터를 통합하기 위해서는 다양한 작업이 필요  
1. 지하철 위치 데이터의 역한글명칭 -> 역명 으로 변경
2. 동일 역명에 대해서 다른 호선을 가진 데이터를 통합
3. 지하철 위치 데이터의 환승역Y좌표 -> 위도 로 변경
4. 지하철 위치 데이터의 환승역X좌표 -> 경도 로 변경

그리고 지하철 위치 데이터에서 필요한 이 세 가지 열의 데이터만 이용해서 지하철 승하차 데이터와 통합

In [ ]:
# 지하철 위치 데이터 로드
df = pd.read_csv('데이터/지하철/지하철역_GEOM (역사마스터).csv')

# '역한글명칭'과 '호선명칭' 기준으로 그룹핑 후 첫 번째 행만 선택
df_unique = df.groupby('역한글명칭', as_index=False).first()

df_dropped = df_unique.drop(columns=['호선명칭'])

# 중복 여부 확인
print(df_dropped.duplicated(subset=['역한글명칭']).sum())

# 확인 및 저장
print(df_dropped)

df_dropped.to_csv('데이터/지하철/지하철역_GEOM (역사마스터).csv', index=False)

전에 가공했던 데이터들이 제대로 가공되었는지 확인  

In [ ]:
# 지하철 데이터 로드
subway_frequency_df = pd.read_csv('데이터/지하철/통합_지하철_승하차_데이터.csv')
subway_location_df = pd.read_csv('데이터/지하철/지하철역_GEOM (역사마스터).csv')

print(len(subway_frequency_df))  # 병합 기준이 되는 원본 데이터
print(len(subway_frequency_df['역명'].unique()))

print(len(subway_location_df))  # 위치 정보 데이터
print(len(subway_location_df['역한글명칭'].unique()))

In [ ]:
# 지하철 데이터 로드
subway_frequency_df = pd.read_csv('데이터/지하철/통합_지하철_승하차_데이터.csv')
subway_location_df = pd.read_csv('데이터/지하철/지하철역_GEOM (역사마스터).csv')

# 위치 데이터 컬럼명 변경: 역명, 위도, 경도
subway_location_df = subway_location_df.rename(columns={
    '역한글명칭': '역명',
    '환승역Y좌표': '위도',
    '환승역X좌표': '경도'
})

# 정확한 역명 기준으로 병합 (left join)
merged_df = pd.merge(
    subway_frequency_df,
    subway_location_df[['역명', '위도', '경도']],
    on='역명',
    how='left'
).reset_index(drop=True)

# 병합된 결과 저장
merged_df.to_csv('데이터/지하철/통합_지하철_데이터.csv', index=False)

# 확인용 출력
print(merged_df.dtypes)
print("\n결측치 확인\n", merged_df.isnull().sum())
print(merged_df)

이때 결측치를 확인하였을 때, 위도와 경도에 대해서 결측치가 발생한 것을 확인  
이를 처리하기 위해 위,경도에 대해 결측치가 있는 역명 목록을 추출함을 통해 추가 분석

In [ ]:
# 위경도 결측치가 있는 역명 목록 추출
missing_geo_stations = merged_df[merged_df['위도'].isna()]['역명'].unique()
missing_geo_stations = sorted(missing_geo_stations)

print(f"위경도 결측 역 개수: {len(missing_geo_stations)}")
print("결측 역명 목록:")
for station in missing_geo_stations:
    print(station)

결측치를 확인했을 때 결측치가 존재했는데 이는 정확히 역명이 일치하지 않았기 때문으로 확인함

결측치 조사 결과

좌측이 승하차 데이터 상 역명, 우측이 역 위치 데이터 상 역명

낙성대(강감찬) -> 낙성대  
동대문역사문화공원(DDP) -> 동대문역사문화공원  
마곡나루(서울식물원) -> 마곡나루  
상봉 -> 상봉(시외버스터미널)  
용마산(용마폭포공원) -> 용마산  
자양(뚝섬한강공원) -> 뚝섬유원지   
평택지제 -> 지제  
하남시청(덕풍·신장) -> 하남시청(덕풍-신장)  
한국항공대 -> 화전  

그리고 이를 이용해 역 위치 데이터상의 역명을 승하차 데이터 상의 역명으로 수동 매핑 후 통합 과정 진행

In [ ]:
# 역 위치 데이터상의 역명 → 승하차 데이터상의 역명 으로 수동 매핑
manual_station_map = {
    "낙성대": "낙성대(강감찬)",
    "동대문역사문화공원": "동대문역사문화공원(DDP)",
    "마곡나루": "마곡나루(서울식물원)",
    "상봉(시외버스터미널)": "상봉",
    "용마산": "용마산(용마폭포공원)",
    "뚝섬유원지": "자양(뚝섬한강공원)",
    "지제": "평택지제",
    "하남시청(덕풍-신장)": "하남시청(덕풍·신장)",
    "화전": "한국항공대"
}

# 수동 매핑용 역 위치 데이터 추출
manual_geo_df = subway_location_df[subway_location_df["역명"].isin(manual_station_map.keys())].copy()

# 역명을 승하차 데이터용 이름으로 교체
manual_geo_df["역명"] = manual_geo_df["역명"].map(manual_station_map)

# 기존 병합된 결과 불러오기 (이미 저장된 파일 기준으로)
merged_df = pd.read_csv('데이터/지하철/통합_지하철_데이터.csv').reset_index(drop=True)

# 기존 결측치 대상 중 수동 매핑 대상만 별도로 필터링
merged_missing = merged_df[merged_df["역명"].isin(manual_geo_df["역명"])]

# 수동 매핑 좌표 보완 merge
merged_filled = pd.merge(
    merged_missing.drop(columns=["위도", "경도"]),  # 기존 위경도 제거 후
    manual_geo_df[["역명", "위도", "경도"]],
    on="역명",
    how="left"
)

# 기존 데이터에서 결측치 대상만 제외하고 → 보완된 데이터 다시 추가
merged_df = pd.concat([
    merged_df[~merged_df["역명"].isin(manual_geo_df["역명"])],
    merged_filled
], ignore_index=True)

# "사용일자"와 "역명" 기준으로 정렬
merged_df = merged_df.sort_values(by=["사용일자", "역명"]).reset_index(drop=True)

# 위경도 결측 여부 최종 확인
final_missing = merged_df[merged_df["위도"].isna()]["역명"].unique()
print(f"보완 후 남은 위경도 결측 역 개수: {len(final_missing)}")

# 저장
merged_df.to_csv('데이터/지하철/통합_지하철_데이터.csv', index=False)

# 확인용 출력
# 사용일자별로 역명 개수 세기
station_count_per_date = merged_df.groupby('사용일자')['역명'].nunique()

# 고유한 역명 개수 리스트 추출
unique_counts = station_count_per_date.unique()

# 결과 출력
if len(unique_counts) == 1:
    print(f"모든 사용일자에 대해 역명 개수가 동일합니다: {unique_counts[0]}개")
else:
    print("사용일자별로 역명 개수가 서로 다릅니다.")

# (사용일자, 역명) 중복 여부 확인
dupes = merged_df.groupby(['사용일자', '역명']).size().reset_index(name='count')
dupes = dupes[dupes['count'] > 1]

if not dupes.empty:
    print(f"\n (사용일자, 역명) 조합 중복 발견: {len(dupes)}건")
    print(dupes.sort_values(['사용일자', '역명']))
else:
    print("\n 모든 (사용일자, 역명) 조합이 고유합니다. 중복 없음.")

print(merged_df.dtypes)
print()
print(merged_df.isnull().sum())
print()
print(merged_df)

결측치 및 사용일자에 대한 확인을 통해 데이터가 의도대로 가공되었음을 확인

### 데이터를 이용하기 위한 통합 데이터 전처리(사용일자 -> 일시, 하차총승객수 -> 인구유입량)

데이터를 나중에 이용하기 위해서 통합 지하철 데이터에 대해서 사용일자 열의 이름을 일시로 변경  
그리고 인구유입량의 경우 보수적으로 설정하기 위해서 하차총승객수를 인구유입량으로 설정 -> 그리고 이를 위해서 인구유입량 열을 생성하고 기존의 승하차승객수 열을 삭제

In [ ]:
df = pd.read_csv('데이터/지하철/통합_지하철_데이터.csv')

# '사용일자' 컬럼을 datetime 형식으로 변환
df['사용일자'] = pd.to_datetime(df['사용일자'], format='%Y%m%d')

# 컬럼명 '사용일자'를 '일시'로 변경
df.rename(columns={'사용일자': '일시'}, inplace=True)

# 인구유입량 컬럼 생성
df['인구유입량'] = df['하차총승객수']

# 기존 승차/하차 컬럼 제거
df.drop(['승차총승객수', '하차총승객수'], axis=1, inplace=True)

df.to_csv('데이터/지하철/통합_지하철_데이터.csv', index=False, encoding='utf-8')

# 결과 확인
print(df)

통합 이후 마지막으로 다시 결함에 대한 부분이 있을 수 있으므로 추가적으로 처리 및 확인을 위한 과정을 진행

In [ ]:
df = pd.read_csv('데이터/지하철/통합_지하철_데이터.csv')

df_dedup = df.drop_duplicates(subset=['일시', '역명'], keep='first').reset_index(drop=True)

df_dedup.to_csv('데이터/지하철/통합_지하철_데이터.csv', index=False, encoding='utf-8')

# 사용일자별로 역명 개수 세기
station_count_per_date = df_dedup.groupby('일시')['역명'].nunique()

# 고유한 역명 개수 리스트 추출
unique_counts = station_count_per_date.unique()

# 결과 출력
if len(unique_counts) == 1:
    print(f"모든 사용일자에 대해 역명 개수가 동일합니다: {unique_counts[0]}개")
else:
    print("사용일자별로 역명 개수가 서로 다릅니다.")

# (일시, 역명) 중복 여부 검사
dupes = df_dedup.groupby(['일시', '역명']).size().reset_index(name='count')
dupes = dupes[dupes['count'] > 1]

if not dupes.empty:
    print(f"\n 중복된 (일시, 역명) 조합 발견: {len(dupes)}건")
    print(dupes.sort_values(['일시', '역명']))
else:
    print("\n 모든 (일시, 역명) 조합이 고유합니다. 중복 없음.")

# 데이터 타입 및 결측치 확인
print("\n데이터 타입 정보:")
print(df_dedup.dtypes)

print("\n결측치 개수:")
print(df_dedup.isnull().sum())

# 데이터프레임 확인
print("\n데이터 미리보기:")
print(df_dedup)

# 일시별 전체 행(row) 개수 세기
row_count_per_date = df_dedup.groupby('일시').size()

print("\n일시별 행(row) 개수:")
print(row_count_per_date.sort_index())

In [ ]:
# 일시별 전체 행(row) 개수 세기
row_count_per_date = df_dedup.groupby('일시').size()

# 고유한 개수만 추출
unique_row_counts = row_count_per_date.unique()

# 결과 출력
if len(unique_row_counts) == 1:
    print(f"\n모든 일자에 대해 행 개수가 동일합니다: {unique_row_counts[0]}개")
else:
    print("\n일자별로 행 개수가 서로 다릅니다.")
    print("개수 분포:")
    print(row_count_per_date.value_counts().sort_index())
    print("\n일시별 행 개수 목록:")
    print(row_count_per_date.sort_index())


# 1. '일시'를 datetime.date 형식으로 변환
df_dedup['일시'] = pd.to_datetime(df_dedup['일시'], errors='coerce').dt.date

# 2. 기대하는 전체 날짜 범위 생성
expected_dates = pd.date_range(start='2022-01-01', end='2024-12-31').date

# 3. 실제 데이터에 존재하는 날짜
actual_dates = set(df_dedup['일시'].dropna().unique())

# 4. 누락된 날짜 추출
missing_dates = sorted(set(expected_dates) - actual_dates)

# 5. 결과 출력
if not missing_dates:
    print("\n모든 일자가 누락 없이 포함되어 있습니다.")
else:
    print(f"\n누락된 일자 수: {len(missing_dates)}")
    print("누락된 일자 목록:")
    for d in missing_dates:
        print(d)

통합 전 승하차 데이터와 비교 했을 때 행의 개수가 동일한 것을 확인  
그리고 결측치 및 모든 데이터가 정상적으로 처리된 것을 확인  
또한 모든 일자에 대해 행 개수, 그리고 모든 일자가 누락 없이 포함, 중복 없이 일자 - 역명 조합이 구성되어 있음을 확인  
이를 마지막으로 지하철 데이터에 대한 전처리를 마무리함

# 휴일 구분 데이터 처리

휴일 구분 데이터 처리는 holidays 라이브러리를 활용하여 실제 날짜별 휴일 구분을 가진 csv 파일 생성

In [ ]:
# 공휴일, 평일, 주말 구분 CSV 생성

# 1. 날짜 범위 설정
start_date = datetime.date(2022, 1, 1)
end_date = datetime.date(2024, 12, 31)

# 2. 대한민국 공휴일 로딩
kr_holidays = holidays.KR(years=[2022, 2023, 2024])

# 3. 평일/휴일 분류 함수
def classify_date(date):
    if date in kr_holidays:
        return "주말/공휴일"
    elif date.weekday() >= 5:  # 토(5), 일(6)
        return "주말/공휴일"
    else:
        return "평일"

# 4. 날짜 리스트 및 분류 결과 저장
date_list = []
classification = []

current_date = start_date
while current_date <= end_date:
    date_list.append(current_date)
    classification.append(classify_date(current_date))
    current_date += datetime.timedelta(days=1)

# 5. DataFrame 생성 및 일시를 datetime64[ns]로 변환
df = pd.DataFrame({
    "일시": date_list,
    "휴일 구분": classification
})

df["일시"] = pd.to_datetime(df["일시"])  # datetime 형식으로 변환

# 6. CSV 파일 저장
df.to_csv('데이터/휴일/휴일_구분_데이터.csv', index=False, encoding='utf-8')

print(df.dtypes)
print("\n결측치 확인\n", df.isnull().sum())
print()
print(df.head())

추가로 방학에 대한 데이터 또한 추가(여름방학: 7월 ~ 8월, 겨울방학: 1월 ~ 2월)

In [ ]:
# 방학 여부 분류 함수
def classify_vacation(date):
    month = date.month
    if month in [1, 2]:
        return '겨울방학'
    elif month in [7, 8]:
        return '여름방학'
    else:
        return 'X'

df = pd.read_csv('데이터/휴일/휴일_구분_데이터.csv', parse_dates=['일시'])

# 방학 여부 컬럼 추가
df['방학 여부'] = df['일시'].apply(classify_vacation)

# CSV 파일 저장
df.to_csv('데이터/휴일/휴일_방학_구분_데이터.csv', index=False, encoding='utf-8')

# 결과 확인용 출력
print(df.dtypes)
print("\n결측치 확인\n", df.isnull().sum())
print()
print(df)

마지막의 결과 확인을 통해서 결측치가 없으며, 1096일에 대한 데이터가 전부 만들어졌음을 확인   
이를 마지막으로 휴일 구분 데이터에 대한 생성 및 처리를 마무리

# 기상 데이터 처리

## 기상 데이터 출처

기상청 일별 기상 데이터: https://data.kma.go.kr/data/grnd/selectAsosRltmList.do?pgmNo=36

## 기상 데이터 통합(전 기간에 대해서)

먼저 기상 데이터의 경우 2022, 2023, 2024년으로 나누어져 있으므로 이에 대해서 먼저 통합을 실시함  
그리고 원본 데이터와 가공을 위한 데이터에 대한 분리를 실시  
이때 앞의 지하철 데이터와 같이 csv 파일별로 다른 경우에 대해서 먼저 확인하기 위해 csv 파일 각각을 확인    
그리고 동일함을 확인하여 통합을 실시

In [ ]:
# 기상 데이터 통합(2022-2024)

# 디렉터리 경로 설정
input_dir = '데이터(보존)/기상/'

# 파일명 조건에 맞는 리스트 생성
file_list = [
    f for f in os.listdir(input_dir)
]

df_list = []
for file in file_list:
    input_path = input_dir + file

    try:
        df = pd.read_csv(input_path)
    except UnicodeDecodeError:
        df = pd.read_csv(input_path, encoding='cp949')


    df_list.append(df)
    print(df)

# 데이터 병합 및 저장
merge_df = pd.concat(df_list, ignore_index=True)
merge_df.to_csv('데이터/기상/통합_기상_데이터.csv', index=False, encoding='utf-8')

# 확인용 출력
print("\n병합 완료. 데이터 타입 및 결측치 확인:\n")
print(merge_df.dtypes)
print("\n결측치 확인\n", merge_df.isnull().sum())
print(merge_df)

1096개 행이 있음을 확인 및 각 열에 대해서 결측치가 있음을 확인  
그러나 결측치의 경우 뒤에 있을 기상 데이터 모델링에 필요한 열만을 필터링한 이후에 처리할 예정

## 데이터를 이용하기 위한 통합 데이터 전처리

기상 데이터를 간단한 모델링을 통해서 두 가지로 구분하려고 함  
1. 계절  
2. 날씨 유형  

이때 계절을 통해 대략적인 기온 및 날씨 정보  
날씨 유형을 통해 세부적인 날씨 정보에 대한 간단한 모델링을 하려고 함  

이러한 모델링을 위해서는 위의 기상 데이터에서 일시, 평균기온, 일강수량, 평균 상대습도, 일 최심적설, 평균 전운량만을 이용하기로 함  

In [ ]:
# 실제 이용 및 데이터 가공에 필요한 Column 선별

# CSV 파일 불러오기
df = pd.read_csv('데이터/기상/통합_기상_데이터.csv')

# 유지할 컬럼 목록
columns_to_keep = [
    '일시',
    '평균기온(°C)',
    '일강수량(mm)',
    '평균 상대습도(%)',
    '일 최심적설(cm)',
    '평균 전운량(1/10)'
]

# 지정된 컬럼만 추출
df_filtered = df[columns_to_keep]

# '일시' 컬럼을 datetime 형식으로 변환
df_filtered['일시'] = pd.to_datetime(df_filtered['일시'])

# 결과 저장 (선택)
df_filtered.to_csv('데이터/기상/통합_기상_데이터.csv', index=False, encoding='utf-8')

# 결과 확인
print(df_filtered.dtypes)
print("\n결측치 확인\n", df_filtered.isnull().sum())
print()
print(df_filtered.head())

그리고 이러한 전처리 후에 결측치를 확인  
결측치 확인 결과 강수량 및 최심적설이 없는 날은 비,눈이 내리지 않은 날로 판단  
0으로 결측치를 채움

In [ ]:
# CSV 파일 불러오기
df = pd.read_csv('데이터/기상/통합_기상_데이터.csv')

# 결측치 0으로 fillna
df = df.fillna(0)

# 확인용 출력
print("\n결측치 확인\n", df.isnull().sum())
print(df.head())

나중에 사용하기 위해 계절 데이터를 추가하기 위해서 기준을 세움  
이 기준의 경우 기상청의 기준을 따름  

1. 겨울: 평균 기온이 5도 미만으로 하강한 후 5일 동안 유지된 첫날부터  
2. 봄: 평균 기온 5도 이상으로 상승한 후 5일 동안 유지된 첫날부터  
3. 여름: 평균 기온 20도 이상으로 상승한 후 5일 동안 유지된 첫날부터  
4. 가을: 평균 기온 20도 미만으로 하강한 후 5일 동안 유지된 첫날부터  

그리고 이를 활용하여 계절 데이터를 생성

In [ ]:
# '일시' 컬럼을 datetime 형식으로 변환
df['일시'] = pd.to_datetime(df['일시'])

# 계절 컬럼 초기화
df['계절'] = None

# 시작 계절 설정
current_season = '겨울'
df.loc[0, '계절'] = current_season

# 계절 판별 함수 (5일 연속 조건 만족 시 계절 전환)
def detect_season_start(condition_series):
    count = 0
    for i in range(len(condition_series)):
        if condition_series[i]:
            count += 1
            if count >= 5:
                return i - 4  # 시작 인덱스
        else:
            count = 0
    return None

# 인덱스 포인터
i = 1

while i < len(df):
    temp_series = df['평균기온(°C)'].iloc[i:]

    # 계절 조건 정의 및 판별
    if current_season == '겨울':
        condition = temp_series >= 5
        start_idx = detect_season_start(condition.values)
        if start_idx is not None:
            season_start = i + start_idx
            df.loc[i:season_start - 1, '계절'] = '겨울'
            current_season = '봄'
            i = season_start
        else:
            df.loc[i:, '계절'] = '겨울'
            break

    elif current_season == '봄':
        condition = temp_series >= 20
        start_idx = detect_season_start(condition.values)
        if start_idx is not None:
            season_start = i + start_idx
            df.loc[i:season_start - 1, '계절'] = '봄'
            current_season = '여름'
            i = season_start
        else:
            df.loc[i:, '계절'] = '봄'
            break

    elif current_season == '여름':
        condition = temp_series < 20
        start_idx = detect_season_start(condition.values)
        if start_idx is not None:
            season_start = i + start_idx
            df.loc[i:season_start - 1, '계절'] = '여름'
            current_season = '가을'
            i = season_start
        else:
            df.loc[i:, '계절'] = '여름'
            break

    elif current_season == '가을':
        condition = temp_series < 5
        start_idx = detect_season_start(condition.values)
        if start_idx is not None:
            season_start = i + start_idx
            df.loc[i:season_start - 1, '계절'] = '가을'
            current_season = '겨울'
            i = season_start
        else:
            df.loc[i:, '계절'] = '가을'
            break

# 결과 저장
df.to_csv('데이터/기상/통합_기상_데이터.csv', index=False, encoding='utf-8')

# 계절별 일수 요약 및 결과 확인
print(df['계절'].value_counts())
print("\n결측치 확인\n", df.isnull().sum())
print()
print(df)

계절 데이터 생성 이후 데이터를 확인하였을 때 문제 없이 생성되었음을 확인  

간단한 날씨 기반 분석을 하기 위한 두 번째 모델링인 날씨 유형 데이터를 추가     

그리고 계절, 날씨 유형 이외의 데이터는 사용하지 않기 때문에 일시, 계절, 날씨 유형을 제외한 나머지 열 제거  

날씨 유형의 경우 아래 나타난 날씨 유형 분류 함수에 따라 분류하였음

In [ ]:
# CSV 파일 불러오기
df = pd.read_csv('데이터/기상/통합_기상_데이터.csv')

# 불쾌지수 계산
# T = 평균기온(°C), RH = 평균 상대습도 (0~1 사이 비율로 변환)
T = df['평균기온(°C)']
RH = df['평균 상대습도(%)'] / 100
df['불쾌지수'] = (9/5)*T - 0.55*(1 - RH)*((9/5)*T - 26) + 32

# 날씨 유형 분류 함수
def classify_weather(row):
    snow = row['일 최심적설(cm)']
    rain = row['일강수량(mm)']
    discomfort = row['불쾌지수']
    cloud = row['평균 전운량(1/10)']

    if snow >= 5:
        return '대설'  # 눈 많이 온날
    elif 0 < snow < 5:
        return '적설'  # 눈 온날
    else:
        if rain >= 3:
            return '강우'  # 비 온날
        elif 1 <= rain < 3:
            return '약우'  # 비 적게 온날
        else:
            if discomfort >= 80:
                return '고습'  # 불쾌지수 높음
            else:
                if cloud < 3:
                    return '맑음'
                elif cloud >= 8:
                    return '흐림'
                else:
                    return '보통'

# 날씨 유형 컬럼 생성
df['날씨 유형'] = df.apply(classify_weather, axis=1)

# 최종 필요한 열만 선택
df_weather = df[['일시', '계절', '날씨 유형']]

# 결과 저장 (선택 사항)
df_weather.to_csv('데이터/기상/통합_기상_데이터.csv', index=False, encoding='utf-8')

# 날씨 유형별 요약 및 결과 확인
print(df_weather['날씨 유형'].value_counts())
print("\n결측치 확인\n", df_weather.isnull().sum())
print()
print(df_weather)

그리고 마지막으로 가공이 완료된 데이터를 확인하였을 때 결측치가 없으며 날씨 유형이 생각한 대로의 분포를 가지고 있음을 확인  

이에 따라 기상 데이터에 대한 전처리를 마무리하고 전체 데이터 통합 과정을 진행

# 지하철/휴일/기상 데이터 통합

## 데이터 통합

전체 데이터를 통합하기 위해서 각 데이터 유형에서 마지막으로 저장한 csv 파일을 로드 및 전처리 과정을 파악하기 위해 데이터를 확인

In [ ]:
# 데이터 로드
subway_df = pd.read_csv('데이터/지하철/통합_지하철_데이터.csv')
holiday_df = pd.read_csv('데이터/휴일/휴일_방학_구분_데이터.csv')
weather_df = pd.read_csv('데이터/기상/통합_기상_데이터.csv')

# 데이터 확인
print(subway_df)
print(holiday_df)
print(weather_df)

데이터 확인 결과 지하철 데이터에 일시를 기준으로 기상 데이터(계절, 날씨 유형), 휴일 구분 데이터(휴일_구분, 방학_여부)를 통합하기로 결정  
그리고 이를 위해 먼저 행의 개수가 같은 날씨, 휴일 데이터를 먼저 병합  
이후 이를 일시를 기준으로 하여 지하철 데이터에 병합

In [ ]:
# 날짜 형식 통일
subway_df['일시'] = pd.to_datetime(subway_df['일시'])
weather_df['일시'] = pd.to_datetime(weather_df['일시'])
holiday_df['일시'] = pd.to_datetime(holiday_df['일시'])

# 날씨, 휴일 데이터 병합
weather_holiday_df = pd.merge(weather_df, holiday_df, on='일시', how='left')

# 지하철 데이터 병합 및 저장
full_df = pd.merge(subway_df, weather_holiday_df, on='일시', how='left')

full_df.to_csv('데이터/통합/통합_데이터.csv', index=False, encoding='utf-8')

# 데이터 확인용 출력
print(full_df.dtypes)
print("\n결측치 확인\n", full_df.isnull().sum())
print()
print(full_df)

데이터 확인용 출력을 통해서 결측치가 없으며, 의도했던 대로 데이터가 생성되었음을 확인하고 데이터 통합 및 전처리를 마무리